# UDiff: Exploratory Analysis

This notebook provides an interactive walkthrough of the **UDiff** deep unfolding
framework for sparse signal recovery. We generate synthetic data, instantiate the
model, run reconstruction, and inspect both quantitative metrics and learned
parameter schedules.

---
## 1. Imports

In [ ]:
import sys
import os

# Ensure the project root is on the path so local packages are importable.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt

from models.model_architecture import UDiff
from utils.preprocessing import (
    generate_sensing_matrix,
    generate_sparse_signal,
    create_measurements,
)
from evaluation.metrics import nmse, nmse_db, psnr, compute_all_metrics

plt.rcParams.update({
    "figure.figsize": (10, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## 2. Configuration

We use deliberately small dimensions so that every cell runs in seconds,
even on a CPU-only machine.

In [ ]:
# Signal / measurement dimensions
N = 64            # Signal length
M = 32            # Number of measurements (compression ratio = M/N = 0.5)
SPARSITY = 0.1    # Fraction of non-zero entries
SNR_DB = 20.0     # Measurement SNR in dB
NUM_SAMPLES = 8   # Batch size for this demo

# Model hyper-parameters (kept small for speed)
NUM_STAGES = 8
IN_CHANNELS = 1
CHANNELS_LIST = [16, 32, 32]
EMBEDDING_DIM = 32
SENSING_MODE = "general"  # 'general' (Gaussian) or 'orthonormal'

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

---
## 3. Data Generation

Generate a batch of synthetic sparse signals and visualise a few examples.

In [ ]:
x_true = generate_sparse_signal(N, SPARSITY, NUM_SAMPLES).to(device)
print(f"Signal shape : {x_true.shape}")
print(f"Non-zeros per signal (approx.): {int(N * SPARSITY)}")

fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for idx, ax in enumerate(axes):
    ax.stem(
        x_true[idx].cpu().numpy(),
        linefmt="C0-",
        markerfmt="C0o",
        basefmt="k-",
        use_line_collection=True,
    )
    ax.set_title(f"Sample {idx}")
    ax.set_xlabel("Index")
axes[0].set_ylabel("Amplitude")
fig.suptitle("Synthetic Sparse Signals", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Sensing Matrix

Create a Gaussian sensing matrix $\Phi \in \mathbb{R}^{M \times N}$ and
inspect its structure.

In [ ]:
Phi = generate_sensing_matrix(M, N, matrix_type="gaussian", seed=SEED).to(device)
print(f"Sensing matrix shape: {Phi.shape}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Heatmap of Phi
im = axes[0].imshow(Phi.cpu().numpy(), aspect="auto", cmap="RdBu_r")
axes[0].set_title("Sensing Matrix $\\Phi$")
axes[0].set_xlabel("Column (signal dim)")
axes[0].set_ylabel("Row (measurement dim)")
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

# Singular value spectrum
svd_vals = torch.linalg.svdvals(Phi).cpu().numpy()
axes[1].bar(range(len(svd_vals)), svd_vals, color="steelblue")
axes[1].set_title("Singular Values of $\\Phi$")
axes[1].set_xlabel("Index")
axes[1].set_ylabel("$\\sigma_i$")

plt.tight_layout()
plt.show()

---
## 5. Create Measurements

Compute noisy measurements $y = \Phi x + n$ where the noise level is
set by the target SNR.

In [ ]:
y = create_measurements(x_true, Phi, snr_db=SNR_DB).to(device)
print(f"Measurement shape: {y.shape}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(y[0].cpu().numpy(), "o-", markersize=3, label="Measurements $y$")
ax.set_title("Measurement Vector (Sample 0)")
ax.set_xlabel("Measurement index")
ax.set_ylabel("Value")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Model Instantiation

Build a UDiff model with the configuration defined above.

In [ ]:
model = UDiff(
    signal_dim="1d",
    num_stages=NUM_STAGES,
    in_channels=IN_CHANNELS,
    channels_list=CHANNELS_LIST,
    sensing_mode=SENSING_MODE,
    embedding_dim=EMBEDDING_DIM,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

---
## 7. Forward Pass (Reconstruction)

Run the model on the synthetic measurements and collect per-stage
intermediate estimates.

In [ ]:
model.eval()
with torch.no_grad():
    x_rec, intermediates = model(y, Phi)

print(f"Reconstructed signal shape: {x_rec.shape}")
print(f"Number of intermediate estimates: {len(intermediates)}")

---
## 8. Visualization: Original vs. Reconstructed

Compare the ground-truth sparse signal with the UDiff reconstruction.

In [ ]:
sample_idx = 0
x_np = x_true[sample_idx].cpu().numpy()
x_rec_np = x_rec[sample_idx].cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

axes[0].stem(
    x_np, linefmt="C0-", markerfmt="C0o", basefmt="k-",
    use_line_collection=True,
)
axes[0].set_title("Ground Truth")
axes[0].set_xlabel("Index")
axes[0].set_ylabel("Amplitude")

axes[1].stem(
    x_rec_np, linefmt="C1-", markerfmt="C1o", basefmt="k-",
    use_line_collection=True,
)
axes[1].set_title("UDiff Reconstruction (untrained)")
axes[1].set_xlabel("Index")

fig.suptitle("Signal Reconstruction Comparison", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 9. Quantitative Metrics

Compute NMSE (linear & dB) and PSNR to quantify reconstruction quality.
Since the model is **untrained**, we expect relatively poor numbers here;
these serve as a baseline.

In [ ]:
metrics = compute_all_metrics(x_rec, x_true)

print("=" * 40)
print("  Reconstruction Metrics (untrained)")
print("=" * 40)
for key, value in metrics.items():
    print(f"  {key:>12s} : {value:.4f}")
print("=" * 40)

---
## 10. Per-Stage Convergence

Track how reconstruction quality improves across the $K$ unfolding stages.

In [ ]:
stage_nmse = []
for k, x_k in enumerate(intermediates):
    val = nmse(x_k, x_true).item()
    stage_nmse.append(val)
    print(f"  Stage {k+1:2d}  |  NMSE = {val:.6f}  |  NMSE(dB) = {10 * np.log10(val + 1e-12):.2f} dB")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(stage_nmse) + 1), stage_nmse, "o-", linewidth=2, markersize=6)
ax.set_xlabel("Unfolding Stage $k$")
ax.set_ylabel("NMSE")
ax.set_title("Per-Stage NMSE Convergence")
ax.set_xticks(range(1, len(stage_nmse) + 1))
plt.tight_layout()
plt.show()

---
## 11. Learned Parameter Schedules

Visualise the per-stage learned parameters $\{\sigma_k\}$, $\{\lambda_k\}$,
and $\{\eta_k\}$. These schedules are initialised randomly and refined
during training.

In [ ]:
# Extract schedule parameters from the model
sigma_vals, lambda_vals, eta_vals = [], [], []

for name, param in model.named_parameters():
    name_lower = name.lower()
    if "sigma" in name_lower:
        sigma_vals.extend(param.detach().cpu().flatten().tolist())
    elif "lambda" in name_lower or "lam" in name_lower:
        lambda_vals.extend(param.detach().cpu().flatten().tolist())
    elif "eta" in name_lower:
        eta_vals.extend(param.detach().cpu().flatten().tolist())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

if sigma_vals:
    axes[0].bar(range(1, len(sigma_vals) + 1), sigma_vals, color="coral")
    axes[0].set_title("Noise Level Schedule $\\sigma_k$")
    axes[0].set_xlabel("Stage $k$")
    axes[0].set_ylabel("$\\sigma_k$")
else:
    axes[0].text(0.5, 0.5, "No sigma params found", ha="center", va="center")

if lambda_vals:
    axes[1].bar(range(1, len(lambda_vals) + 1), lambda_vals, color="steelblue")
    axes[1].set_title("Consistency Weight $\\lambda_k$")
    axes[1].set_xlabel("Stage $k$")
    axes[1].set_ylabel("$\\lambda_k$")
else:
    axes[1].text(0.5, 0.5, "No lambda params found", ha="center", va="center")

if eta_vals:
    axes[2].bar(range(1, len(eta_vals) + 1), eta_vals, color="seagreen")
    axes[2].set_title("Relaxation Schedule $\\eta_k$")
    axes[2].set_xlabel("Stage $k$")
    axes[2].set_ylabel("$\\eta_k$")
else:
    axes[2].text(0.5, 0.5, "No eta params found", ha="center", va="center")

fig.suptitle("Learned Per-Stage Parameter Schedules (Untrained Initialisation)", fontweight="bold")
plt.tight_layout()
plt.show()

---

## Summary

This notebook demonstrated the end-to-end pipeline of the UDiff framework:

1. **Data generation** — synthetic sparse signals with configurable sparsity.
2. **Sensing** — Gaussian measurement matrix and noisy measurement creation.
3. **Model** — UDiff with $K$ unfolding stages, shared denoiser, and
   measurement consistency projection.
4. **Reconstruction** — forward-pass reconstruction from compressed measurements.
5. **Evaluation** — NMSE, PSNR, and per-stage convergence tracking.
6. **Schedules** — inspection of learned $\sigma_k$, $\lambda_k$, $\eta_k$.

After training, re-run this notebook to observe improved reconstruction
quality and meaningful parameter schedules.